In [159]:
import pandas as pd
import numpy as np
import os
from scipy.optimize import minimize

# ============================================================
# CONFIGURATION
# ============================================================
DATA_DIR = "Data_2026"
OUTPUT_DIR = "Data_cleaned"
REGION     = "AMER"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [160]:
# ============================================================
# CELLULE 2 — CHARGEMENT BRUT
# ============================================================

# Static
static = pd.read_excel(os.path.join(DATA_DIR, "Static_2025.xlsx"))
static.columns = static.columns.str.strip()
static = static[static['Region'] == REGION].copy()
static = static.set_index('ISIN')
print(f"✅ Static chargé : {len(static)} firmes dans '{REGION}'")

# CO2 Scope 1 & 2
s1 = pd.read_excel(os.path.join(DATA_DIR, "DS_CO2_SCOPE_1_Y_2025.xlsx"), index_col='ISIN')
s2 = pd.read_excel(os.path.join(DATA_DIR, "DS_CO2_SCOPE_2_Y_2025.xlsx"), index_col='ISIN')
print(f"✅ CO2 S1 chargé : {s1.shape} | S2 : {s2.shape}")

# Revenue
rev = pd.read_excel(os.path.join(DATA_DIR, "DS_REV_Y_2025.xlsx"), index_col='ISIN')
print(f"✅ Revenue chargé : {rev.shape}")

# Market Value mensuel
mv_m = pd.read_excel(os.path.join(DATA_DIR, "DS_MV_T_USD_M_2025.xlsx"), index_col='ISIN')
print(f"✅ MV mensuel chargé : {mv_m.shape}")

# Market Value annuel
mv_y = pd.read_excel(os.path.join(DATA_DIR, "DS_MV_T_USD_Y_2025.xlsx"), index_col='ISIN')
print(f"✅ MV annuel chargé : {mv_y.shape}")

# Prix mensuels
prices = pd.read_excel(os.path.join(DATA_DIR, "DS_RI_T_USD_M_2025.xlsx"), index_col='ISIN')
print(f"✅ Prix chargé : {prices.shape}")

✅ Static chargé : 669 firmes dans 'AMER'
✅ CO2 S1 chargé : (2609, 28) | S2 : (2609, 28)
✅ Revenue chargé : (2609, 28)
✅ MV mensuel chargé : (2609, 315)
✅ MV annuel chargé : (2609, 28)
✅ Prix chargé : (2609, 315)


In [161]:
# ============================================================
# CELLULE 3 — MISSING PRICES
# Supprimer les firmes sans aucune donnée (lignes entièrement vides)
# ============================================================

# Garder uniquement les colonnes numériques
s1 = s1.select_dtypes(include=[np.number])
s2 = s2.select_dtypes(include=[np.number])
rev = rev.select_dtypes(include=[np.number])
mv_m = mv_m.select_dtypes(include=[np.number])
mv_y = mv_y.select_dtypes(include=[np.number])
prices = prices.select_dtypes(include=[np.number])

# Supprimer les firmes sans aucune donnée
s1     = s1.dropna(how='all')
s2     = s2.dropna(how='all')
rev    = rev.dropna(how='all')
mv_m   = mv_m.dropna(how='all')
mv_y   = mv_y.dropna(how='all')
prices = prices.dropna(how='all')

print(f"✅ Après suppression lignes vides :")
print(f"   CO2 S1  : {s1.shape}")
print(f"   CO2 S2  : {s2.shape}")
print(f"   Revenue : {rev.shape}")
print(f"   MV mens : {mv_m.shape}")
print(f"   MV ann  : {mv_y.shape}")
print(f"   Prix    : {prices.shape}")

✅ Après suppression lignes vides :
   CO2 S1  : (2394, 26)
   CO2 S2  : (2396, 26)
   Revenue : (2528, 26)
   MV mens : (2543, 313)
   MV ann  : (2542, 26)
   Prix    : (2543, 313)


In [162]:
# ============================================================
# CELLULE 4 — MISSING VALUES
# ============================================================

# ── CO2 : S1 + S2 ────────────────────────────────────────
common_cols = s1.columns.intersection(s2.columns)
s1 = s1[common_cols]
s2 = s2[common_cols]

co2 = s1.add(s2, fill_value=0)
co2[s1.isna() & s2.isna()] = np.nan

# Forward-fill (milieu et fin de sample)
co2 = co2.ffill(axis=1)
print(f"✅ CO2 (S1+S2) : {co2.shape}")

# ── Revenue : forward-fill ────────────────────────────────
rev = rev.ffill(axis=1)
print(f"✅ Revenue forward-fillé : {rev.shape}")

# ── Prix : délisting (fin de sample → return -100%) ──────
def apply_delisting(row):
    last = row.last_valid_index()
    if last is None:
        return row
    after = row[row.index > last]
    if len(after) > 0:
        row = row.copy()
        row.iloc[row.index.get_loc(after.index[0])] = 0.0
    return row

# Convertir les colonnes en datetime
prices.columns = pd.to_datetime(prices.columns, errors='coerce')
prices = prices.loc[:, prices.columns.notna()]
mv_m.columns   = pd.to_datetime(mv_m.columns, errors='coerce')
mv_m = mv_m.loc[:, mv_m.columns.notna()]

prices = prices.apply(apply_delisting, axis=1)
print(f"✅ Délisting appliqué")

✅ CO2 (S1+S2) : (2402, 26)
✅ Revenue forward-fillé : (2528, 26)
✅ Délisting appliqué


In [163]:
# ============================================================
# CELLULE 5 — LOW PRICES
# Prix < 0.5 traités comme NaN
# ============================================================

low_mask = (prices > 0) & (prices < 0.5)
print(f"   Prix < 0.5 détectés : {low_mask.sum().sum()} cellules")
prices[low_mask] = np.nan

print(f"✅ Low prices traités : {prices.shape}")

   Prix < 0.5 détectés : 24692 cellules
✅ Low prices traités : (2543, 313)


In [164]:
# ============================================================
# CELLULE 6 — STALE PRICES
# Exclure les firmes avec > 50% de returns = 0
# sur la fenêtre de 10 ans
# ============================================================

# Calcul des returns
returns = prices.pct_change(axis=1, fill_method=None)
returns = returns.clip(lower=-1.0)

def get_stale_firms(returns_df, year_Y, window=10, threshold=0.50):
    start = year_Y - window + 1
    mask  = ((returns_df.columns.year >= start) &
             (returns_df.columns.year <= year_Y))
    window_ret = returns_df.loc[:, mask]

    if window_ret.empty:
        return pd.Index([])

    prop_zero = (window_ret == 0).sum(axis=1) / window_ret.shape[1]
    return prop_zero[prop_zero > threshold].index

# Exemple : vérification sur 2013 (première année d'allocation)
stale_2013 = get_stale_firms(returns, year_Y=2013)
print(f"✅ Returns calculés : {returns.shape}")
print(f"   Firmes stale en 2013 : {len(stale_2013)}")
print(f"   (ces firmes seront exclues dynamiquement à chaque année)")

✅ Returns calculés : (2543, 313)
   Firmes stale en 2013 : 0
   (ces firmes seront exclues dynamiquement à chaque année)


In [165]:
# ============================================================
# CELLULE 7 — UNIVERS FINAL, VÉRIFICATIONS ET EXPORT
# ============================================================

# ── Intersection de tous les datasets ────────────────────
common = (static.index
          .intersection(prices.index)
          .intersection(co2.index)
          .intersection(rev.index)
          .intersection(mv_m.index))

# Exclure firmes sans aucune donnée CO2
has_co2 = co2.loc[common].dropna(how='all').index
common  = common.intersection(has_co2)

print(f"📊 ISIN communs : {len(common)} firmes")

# ── Appliquer le filtre ───────────────────────────────────
static_cl  = static.loc[common]
prices_cl  = prices.loc[common]
returns_cl = returns.loc[common]
co2_cl     = co2.loc[common]
rev_cl     = rev.loc[common]
mv_m_cl    = mv_m.loc[mv_m.index.isin(common)]
mv_y_cl    = mv_y.loc[mv_y.index.isin(common)]

# ── 2ème filtre : garantir aucune ligne vide ──────────────
common_final = (static_cl.index
                .intersection(co2_cl.dropna(how='all').index)
                .intersection(rev_cl.dropna(how='all').index)
                .intersection(prices_cl.dropna(how='all').index))

if len(common_final) < len(common):
    print(f"   2ème filtre : {len(common) - len(common_final)} firmes supprimées")

static_cl  = static_cl.loc[common_final]
prices_cl  = prices_cl.loc[common_final]
returns_cl = returns_cl.loc[common_final]
co2_cl     = co2_cl.loc[common_final]
rev_cl     = rev_cl.loc[common_final]
mv_m_cl    = mv_m_cl.loc[mv_m_cl.index.isin(common_final)]
mv_y_cl    = mv_y_cl.loc[mv_y_cl.index.isin(common_final)]

# ── Vérifications ─────────────────────────────────────────
print("\n=== VÉRIFICATIONS ===")
print(f"  Prix < 0            : {(prices_cl < 0).sum().sum()}       (doit être 0)")
print(f"  Firmes sans CO2     : {co2_cl.isna().all(axis=1).sum()}    (doit être 0)")
print(f"  Firmes sans Revenue : {rev_cl.isna().all(axis=1).sum()}    (doit être 0)")
print(f"  Firmes sans Prix    : {prices_cl.isna().all(axis=1).sum()} (doit être 0)")
print(f"\n✅ Univers final : {len(common_final)} firmes")

# ── Export Excel ──────────────────────────────────────────
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Data_cleaned.xlsx"), engine='openpyxl') as writer:
    static_cl.to_excel(writer,  sheet_name='Static')
    co2_cl.to_excel(writer,     sheet_name='CO2')
    rev_cl.to_excel(writer,     sheet_name='Revenue')
    prices_cl.to_excel(writer,  sheet_name='Prices')
    returns_cl.to_excel(writer, sheet_name='Returns')
    mv_m_cl.to_excel(writer,    sheet_name='MV_monthly')
    mv_y_cl.to_excel(writer,    sheet_name='MV_annual')

print(f"✅ Data_cleaned.xlsx exporté")

📊 ISIN communs : 594 firmes

=== VÉRIFICATIONS ===
  Prix < 0            : 0       (doit être 0)
  Firmes sans CO2     : 0    (doit être 0)
  Firmes sans Revenue : 0    (doit être 0)
  Firmes sans Prix    : 0 (doit être 0)

✅ Univers final : 594 firmes
✅ Data_cleaned.xlsx exporté
